# Why lazy evaluation?

You might wonder why LSDB loads catalogs lazily. Why not just download the data? Let's explore this with a metaphor.

### A thoughtful gift

Imagine it's your birthday. Your friend shows up to your party in a moving truck. After greeting you, she says, "Happy birthday! My gift to you is a houseplant, but I wasn't sure which one you would like."

She continues, "Inside this truck is every plant they had at the store. All you have to do is help me move all these plants inside your house so you can choose the ones you want. Then whatever you don't want, you can throw away! Also, I'm renting this truck by the hour, so can you stop what you're doing and help me bring these all in?"

### A forest of stars






The plants in this story represent the data available through the LSST. Over a decade of the survey, dozens or hundreds of petabytes of data will be generated---a vast forest of information compared to your friend's moving truck full of plants. Lazy evaluation is a practical way to handle this volume of data.

### The catalog

Imagine that your friend arrives instead carrying a small booklet, *World of Plants* on its glossy cover. A catalog.

As you open it, she says, "I want to get you a plant. Take a look through this catalog. You can see each plant's dimensions, weight, color, and care instructions, and decide which ones you want for your home. Once you pick out which plants you want, I'll schedule a time to help you move them in."

You and your friend enjoy the rest of the party. A few days later, she helps you move a gorgeous Bird of Paradise plant into your living room.

### Lazy evaluation

This is the principle of lazy evaluation. When a catalog is loaded lazily, no data has been loaded, only the catalog schema. You can refine your search, ensuring that your query will select the data you want, all before you ever download it. Finally, you can compute the catalog to get the objects that match your query.

### LSDB example

Let's use lazy evaluation in LSDB.

In [ ]:
# Setup

import lsdb

import astropy.units as u
from astropy.coordinates import SkyCoord
from upath import UPath
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

from dask.distributed import Client

client = Client(n_workers=4, memory_limit="4GiB", threads_per_worker=1)

# Path on RSP
base_path = UPath("/rubin/lsdb_data")
# Path on epyc
base_path = UPath("/astro/store/shire/hats/dash/hats/dp2/")

In [ ]:
# Load Rubin catalog
cat = lsdb.open_catalog(base_path / "object_collection")

We have loaded the Rubin catalog. This loading is lazy, so it takes only a few seconds.

Now we see all the columns available, the number of partitions, and an estimated total size of the data.

We can also see our catalog's coverage in the sky.

In [ ]:
cat

In [ ]:
cat.plot_pixels();

At this point, we may realize that we only need a subset of this data:

- A certain set of columns
- A certain region in the sky
- A certain filter based on column values

Because of lazy evaluation, we were able to see that we had too much data, and can filter it appropriately.

In [ ]:
cat = (
    lsdb.open_catalog(
        base_path / "object_collection",
        # Select columns
        columns=["g_psfMag", "r_psfMag"],
        # Select a sky region
    )
    .cone_search(
        ra=270.0,
        dec=-30.0,
        radius_arcsec=2 * 3600,
        # Query on column values
    )
    .query("g_psfMag < 28.0 and r_psfMag < 28.0")
)

In [ ]:
cat

Our lazy catalog took only a few seconds to filter. Now we know that we will get only the data we need.

Finally, we can compute the result.

In [ ]:
df = cat.compute()

In [ ]:
df

In [ ]:
# Clean up
client.close()